# Dispute Case Tracker
Gmail → SQLite → Google Sheets → Slack pipeline for `account.receivable.vn@grabtaxi.com`

**Auth:** OAuth 2.0 (Desktop app). First run opens a browser once to approve access; all subsequent runs are fully headless.

**Sections:**
1. Imports & Configuration
2. Queue Classification
3. Database
4. Gmail Auth & Helpers
5. Core Polling
6. Archive
7. Slack Notifications
7.5. Google Sheets Sync
8. Actions (Assign / Complete / Reclassify)
9. Run: Poll Once
10. Run: List Cases
11. Diagnostics
12. Tests

---
## 1. Imports & Configuration

In [49]:
#install requests package
import sys
!{sys.executable} -m pip install requests

In [50]:
#import requests and print its file path to confirm installation
import requests
print(requests.__file__)

c:\Users\vy.nguyenth\miniconda3\envs\vscode-env\Lib\site-packages\requests\__init__.py


In [51]:
#checking kernel executable and requests import to troubleshoot import error

import sys
print("Kernel executable:", sys.executable)
try:
    import requests
    print("requests path:", requests.__file__)
except Exception as e:
    print("Import error:", type(e).__name__, e)

Kernel executable: c:\Users\vy.nguyenth\miniconda3\envs\vscode-env\python.exe
requests path: c:\Users\vy.nguyenth\miniconda3\envs\vscode-env\Lib\site-packages\requests\__init__.py


In [ ]:
#import libraries and configurations for Gmail API authentication and email processing

import os
import sys
import time
import logging
import base64
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

print("Imports OK")

In [ ]:
# ── Edit these values before running ──────────────────────────────────────────
CONFIG = {
    "GROUP_EMAIL": "account.receivable.vn@grabtaxi.com",
    "PROCESSED_LABEL": "dispute-tracker-processed",
    "POLL_INTERVAL_MINUTES": 5,
    "ARCHIVE_AFTER_DAYS": 1,
    "MAX_RESULTS": 100,
    "TIMEOUT_BUFFER_SECONDS": 5 * 60,
    "CREDENTIALS_FILE": os.path.join(os.getcwd(), "credentials.json"),
    "TOKEN_FILE": os.path.join(os.getcwd(), "token.json"),
    "SHEET_ID": "1tdTFveOGwKRm_d8W_1QyJuM7j1Pn_HioSRC3HhOCmd0",
    "SHEET_TAB_NAME": "Cases",
    "ARCHIVE_TAB_NAME": "Archive",
    "TIMEZONE": "Asia/Ho_Chi_Minh",
}

# Set your Slack webhook URL here, or set the env var SLACK_WEBHOOK_URL before launching Jupyter
SLACK_WEBHOOK_URL = ("https://hooks.slack.com/triggers/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", "")

SCOPES = [
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/spreadsheets",
]
TZ = ZoneInfo(CONFIG["TIMEZONE"])

print("Config loaded")
print(f"  Credentials: {CONFIG['CREDENTIALS_FILE']}")
print(f"  Token:       {CONFIG['TOKEN_FILE']}")
print(f"  Sheet ID:    {CONFIG['SHEET_ID']}")
print(f"  Slack set:   {'Yes' if SLACK_WEBHOOK_URL else 'No'}")

---
## 2. Queue Classification

In [54]:
QUEUES = [
    {
        "name": "Internal Invoice",
        "keywords": ["internal invoice"],
        "match_subject_only": True,
    },
    {
        "name": "Dispute",
        "keywords": ["chênh lệch", "sai sót", "bảng kê", "thiếu", "biên bản điều chỉnh", "dispute","Đối chiếu","sai lệch","cấn trừ"],
        "match_subject_only": False,
    },
    {
        "name": "Update Details",
        "keywords": ["thông tin", "không chính xác", "thay đổi", "update details", "update info"],
        "match_subject_only": False,
    },
    {
        "name": "Invoice",
        "keywords": ["Request invoice", "xuất hóa đơn", "chưa nhận được hóa đơn", "invoice request" ,"cung cấp hóa đơn","cung cấp"],
        "match_subject_only": False,
    },
]

def classify_email(subject: str, body: str) -> str:
    subject_lower = subject.lower()
    full_text = (subject + " " + body).lower()
    for queue in QUEUES:
        haystack = subject_lower if queue["match_subject_only"] else full_text
        for kw in queue["keywords"]:
            if kw.lower() in haystack:
                return queue["name"]
    return "Others"

print("classify_email defined")

classify_email defined


---
## 3. Google Sheets — Database Helpers

Google Sheets is the sole database. All reads and writes go directly to the sheet.

In [ ]:
SHEET_COLUMNS = [
    "Case ID", "Date Received", "Sender", "Subject", "Queue",
    "Status", "Assigned To", "Assigned At", "Completed At", "Email Link",
]

def get_sheets_service():
    return build("sheets", "v4", credentials=_get_or_refresh_creds())

def _get_existing_case_ids() -> set:
    """Read Case ID column from the Cases tab to find already-processed cases."""
    try:
        service = get_sheets_service()
        result = service.spreadsheets().values().get(
            spreadsheetId=CONFIG["SHEET_ID"],
            range=f"{CONFIG['SHEET_TAB_NAME']}!A:A",
        ).execute()
        values = result.get("values", [])
        # Skip header row, collect all case IDs
        return {row[0] for row in values[1:] if row}
    except Exception as e:
        log.error(f"Failed to load existing case IDs from sheet: {e}")
        return set()

def _get_existing_message_ids_from_sheet() -> set:
    """Read Email Link column (col J) to extract message IDs already in the sheet."""
    try:
        service = get_sheets_service()
        result = service.spreadsheets().values().get(
            spreadsheetId=CONFIG["SHEET_ID"],
            range=f"{CONFIG['SHEET_TAB_NAME']}!J:J",
        ).execute()
        values = result.get("values", [])
        msg_ids = set()
        for row in values[1:]:
            if row and row[0]:
                # Email link format: https://mail.google.com/mail/u/0/#inbox/<msg_id>
                msg_ids.add(row[0].split("/")[-1])
        return msg_ids
    except Exception as e:
        log.error(f"Failed to load existing message IDs from sheet: {e}")
        return set()

def generate_case_id() -> str:
    now = datetime.now(TZ)
    return f"CASE-{now.strftime('%Y%m%d')}-{str(int(time.time() * 1000))[-5:]}"

def append_cases_to_sheet(rows: list[dict]):
    """Append new case rows directly to the Cases tab."""
    if not rows:
        return
    service = get_sheets_service()
    values = [
        [
            r["case_id"], r["date_received"], r["sender"], r["subject"],
            r["queue"], "New", "", "", "", r["email_link"],
        ]
        for r in rows
    ]
    service.spreadsheets().values().append(
        spreadsheetId=CONFIG["SHEET_ID"],
        range=f"{CONFIG['SHEET_TAB_NAME']}!A:J",
        valueInputOption="RAW",
        insertDataOption="INSERT_ROWS",
        body={"values": values},
    ).execute()
    log.info(f"Appended {len(rows)} new case(s) to sheet.")

def find_case_row(service, case_id: str) -> int | None:
    """Return 1-based row number of case_id in the Cases tab, or None."""
    result = service.spreadsheets().values().get(
        spreadsheetId=CONFIG["SHEET_ID"],
        range=f"{CONFIG['SHEET_TAB_NAME']}!A:A",
    ).execute()
    col_a = [r[0] if r else "" for r in result.get("values", [])]
    try:
        return col_a.index(case_id) + 1
    except ValueError:
        return None

def update_case(case_id: str, **fields):
    """Update one or more columns for a case in the Cases tab."""
    if not fields:
        return
    col_map = {
        "status": "F", "assigned_to": "G", "assigned_at": "H",
        "completed_at": "I", "queue": "E",
    }
    try:
        service = get_sheets_service()
        row_num = find_case_row(service, case_id)
        if row_num is None:
            log.error(f"update_case: {case_id} not found in sheet")
            return
        data = []
        for field, value in fields.items():
            col = col_map.get(field)
            if col:
                data.append({
                    "range": f"{CONFIG['SHEET_TAB_NAME']}!{col}{row_num}",
                    "values": [[str(value)]],
                })
        if data:
            service.spreadsheets().values().batchUpdate(
                spreadsheetId=CONFIG["SHEET_ID"],
                body={"valueInputOption": "RAW", "data": data},
            ).execute()
            log.info(f"Updated {case_id}: {list(fields.keys())}")
    except Exception as e:
        log.error(f"update_case failed for {case_id}: {e}")

print("Sheet database helpers defined")

---
## 4. Gmail Auth & Helpers

**How auth works:**
- First run → browser opens once for you to approve Gmail + Sheets access → `token.json` saved
- All subsequent runs → silent headless auto-refresh, no browser
- If `token.json` is deleted → browser opens again once

In [56]:
def _get_or_refresh_creds() -> Credentials:
    """Load token.json; refresh silently if expired; run browser flow if missing."""
    creds = None
    token_file = CONFIG["TOKEN_FILE"]
    if os.path.exists(token_file):
        creds = Credentials.from_authorized_user_file(token_file, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                CONFIG["CREDENTIALS_FILE"], SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open(token_file, "w") as f:
            f.write(creds.to_json())
    return creds

def get_gmail_service():
    return build("gmail", "v1", credentials=_get_or_refresh_creds())

def get_sheets_service():
    return build("sheets", "v4", credentials=_get_or_refresh_creds())

def get_or_create_label(service, name: str) -> str:
    labels = service.users().labels().list(userId="me").execute().get("labels", [])
    for label in labels:
        if label["name"] == name:
            return label["id"]
    created = service.users().labels().create(
        userId="me",
        body={"name": name, "labelListVisibility": "labelShow", "messageListVisibility": "show"},
    ).execute()
    return created["id"]

def apply_label(service, message_id: str, label_id: str):
    service.users().messages().modify(
        userId="me",
        id=message_id,
        body={"addLabelIds": [label_id]},
    ).execute()

def get_message_body(payload: dict) -> str:
    if payload.get("mimeType") == "text/plain":
        data = payload.get("body", {}).get("data", "")
        return base64.urlsafe_b64decode(data + "==").decode("utf-8", errors="replace") if data else ""
    for part in payload.get("parts", []):
        text = get_message_body(part)
        if text:
            return text
    return ""

print("Auth helpers defined")
print("First call to get_gmail_service() or get_sheets_service() will open browser if token.json is missing/expired")

Auth helpers defined
First call to get_gmail_service() or get_sheets_service() will open browser if token.json is missing/expired


---
## 5. Core Polling Logic

In [ ]:
def check_new_emails():
    log.info("=== Polling Gmail ===")
    start = time.time()

    try:
        service = get_gmail_service()
    except Exception as e:
        log.error(f"Gmail auth failed: {e}")
        return

    label_id = get_or_create_label(service, CONFIG["PROCESSED_LABEL"])
    existing_ids = _get_existing_message_ids_from_sheet()
    log.info(f"Loaded {len(existing_ids)} existing message IDs from sheet.")

    query = f"to:{CONFIG['GROUP_EMAIL']} -label:{CONFIG['PROCESSED_LABEL']}"
    try:
        result = service.users().messages().list(
            userId="me", q=query, maxResults=CONFIG["MAX_RESULTS"]
        ).execute()
    except Exception as e:
        log.error(f"Gmail search failed: {e}")
        return

    messages = result.get("messages", [])
    log.info(f"Found {len(messages)} unprocessed messages.")

    new_rows, slack_queue, labeled_ids = [], [], []
    skipped = 0

    for msg_ref in messages:
        if time.time() - start > CONFIG["TIMEOUT_BUFFER_SECONDS"]:
            log.warning("Timeout buffer reached. Stopping early.")
            break

        msg_id = msg_ref["id"]
        if msg_id in existing_ids:
            skipped += 1
            continue

        try:
            msg = service.users().messages().get(
                userId="me", id=msg_id, format="full"
            ).execute()
        except Exception as e:
            log.warning(f"Could not fetch message {msg_id}: {e}")
            continue

        headers = {h["name"].lower(): h["value"] for h in msg.get("payload", {}).get("headers", [])}
        subject = headers.get("subject", "(no subject)")
        sender  = headers.get("from", "")
        date_ms = int(msg.get("internalDate", 0))
        date    = datetime.fromtimestamp(date_ms / 1000, tz=TZ)
        body    = get_message_body(msg.get("payload", {}))
        queue   = classify_email(subject, body)
        case_id = generate_case_id()
        email_link = f"https://mail.google.com/mail/u/0/#inbox/{msg_id}"

        row = {
            "case_id":      case_id,
            "message_id":   msg_id,
            "date_received": date.strftime("%d/%m/%Y %H:%M"),
            "sender":       sender,
            "subject":      subject,
            "queue":        queue,
            "email_link":   email_link,
        }
        new_rows.append(row)
        slack_queue.append(row)
        existing_ids.add(msg_id)
        labeled_ids.append(msg_id)

    if new_rows:
        append_cases_to_sheet(new_rows)

    for mid in labeled_ids:
        try:
            apply_label(service, mid, label_id)
        except Exception as e:
            log.warning(f"Could not label {mid}: {e}")

    send_slack_batch(slack_queue)

    duration = time.time() - start
    log.info(f"=== Done in {duration:.1f}s | New: {len(new_rows)} | Skipped: {skipped} ===")

print("check_new_emails defined")

---
## 6. Archive

In [ ]:
def _parse_completed_at(raw: str):
    if not raw:
        return None
    try:
        return datetime.strptime(raw, "%d/%m/%Y %H:%M").replace(tzinfo=TZ)
    except ValueError:
        pass
    try:
        return datetime.fromisoformat(raw).replace(tzinfo=TZ)
    except ValueError:
        return None

def archive_old_completed():
    log.info("=== Archive run ===")
    cutoff = datetime.now(TZ) - timedelta(days=CONFIG["ARCHIVE_AFTER_DAYS"])

    try:
        service = get_sheets_service()
        sheet_id = CONFIG["SHEET_ID"]
        cases_tab = CONFIG["SHEET_TAB_NAME"]
        archive_tab = CONFIG["ARCHIVE_TAB_NAME"]

        result = service.spreadsheets().values().get(
            spreadsheetId=sheet_id,
            range=f"{cases_tab}!A:J",
        ).execute()
        values = result.get("values", [])
        if len(values) < 2:
            log.info("No cases in sheet.")
            return

        header = values[0]
        status_col = header.index("Status") if "Status" in header else 5
        completed_col = header.index("Completed At") if "Completed At" in header else 8

        # Find rows to archive (walk in reverse so row deletions don't shift indices)
        to_archive = []
        for i, row in enumerate(values[1:], start=2):  # 1-based, skip header
            status = row[status_col] if len(row) > status_col else ""
            completed_raw = row[completed_col] if len(row) > completed_col else ""
            if status != "Completed":
                continue
            completed_dt = _parse_completed_at(completed_raw)
            if completed_dt and completed_dt < cutoff:
                to_archive.append((i, row))

        if not to_archive:
            log.info("No completed cases old enough to archive.")
            return

        # Get sheet tab IDs
        meta = service.spreadsheets().get(spreadsheetId=sheet_id).execute()
        sheet_tabs = {s["properties"]["title"]: s["properties"]["sheetId"] for s in meta.get("sheets", [])}

        # Ensure archive tab exists
        if archive_tab not in sheet_tabs:
            service.spreadsheets().batchUpdate(
                spreadsheetId=sheet_id,
                body={"requests": [{"addSheet": {"properties": {"title": archive_tab}}}]},
            ).execute()
            meta = service.spreadsheets().get(spreadsheetId=sheet_id).execute()
            sheet_tabs = {s["properties"]["title"]: s["properties"]["sheetId"] for s in meta.get("sheets", [])}
            # Write header to new archive tab
            service.spreadsheets().values().update(
                spreadsheetId=sheet_id,
                range=f"{archive_tab}!A1",
                valueInputOption="RAW",
                body={"values": [header]},
            ).execute()

        cases_sheet_id = sheet_tabs[cases_tab]

        # Append rows to archive tab
        rows_data = [row[:] + [""] * (len(header) - len(row)) for _, row in to_archive]
        service.spreadsheets().values().append(
            spreadsheetId=sheet_id,
            range=f"{archive_tab}!A:J",
            valueInputOption="RAW",
            insertDataOption="INSERT_ROWS",
            body={"values": rows_data},
        ).execute()

        # Delete from cases tab in reverse order so row numbers stay valid
        delete_requests = []
        for row_num, _ in sorted(to_archive, key=lambda x: x[0], reverse=True):
            delete_requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": cases_sheet_id,
                        "dimension": "ROWS",
                        "startIndex": row_num - 1,
                        "endIndex": row_num,
                    }
                }
            })
        service.spreadsheets().batchUpdate(
            spreadsheetId=sheet_id,
            body={"requests": delete_requests},
        ).execute()

        log.info(f"Archived {len(to_archive)} case(s) to '{archive_tab}' tab.")

    except Exception as e:
        log.error(f"archive_old_completed failed: {e}")

print("archive_old_completed defined")

---
## 7. Slack Notifications

In [60]:
QUEUE_EMOJI = {
    "Dispute": "🚨",
    "Update Details": "📝",
    "Invoice": "🧾",
    "Internal Invoice": "🏢",
    "Others": "📨",
}

def send_slack_raw(text: str):
    if not SLACK_WEBHOOK_URL:
        log.warning("SLACK_WEBHOOK_URL not set — skipping Slack notification.")
        return
    try:
        resp = requests.post(SLACK_WEBHOOK_URL, json={"text": text}, timeout=10)
        if resp.status_code != 200:
            log.warning(f"Slack webhook returned {resp.status_code}: {resp.text}")
    except Exception as e:
        log.error(f"Slack webhook error: {e}")

def send_slack_notification(case: dict):
    emoji = QUEUE_EMOJI.get(case["queue"], "📨")
    text = (
        f"{emoji} *New {case['queue']} Case*\n"
        f"*Case ID:* {case['case_id']}\n"
        f"*From:* {case['sender']}\n"
        f"*Subject:* {case['subject']}\n"
        f"*Email:* {case['email_link']}"
    )
    send_slack_raw(text)

def send_slack_batch(items: list[dict]):
    if not items:
        return
    if len(items) <= 5:
        for item in items:
            send_slack_notification(item)
            time.sleep(0.2)
        return
    by_queue: dict[str, int] = {}
    for item in items:
        by_queue[item["queue"]] = by_queue.get(item["queue"], 0) + 1
    lines = [f"📥 *{len(items)} new cases received*", "━" * 24]
    for q, count in by_queue.items():
        lines.append(f"{QUEUE_EMOJI.get(q, '📨')} *{q}:* {count}")
    lines += ["", "First few cases:"]
    for item in items[:5]:
        lines.append(f"• `{item['case_id']}` - {item['subject']}")
    if len(items) > 5:
        lines.append(f"...and {len(items) - 5} more.")
    send_slack_raw("\n".join(lines))

print("Slack helpers defined")

Slack helpers defined


---
## 7.5. Sheet Utilities

Helper to reload and display current cases from the sheet.

In [ ]:
# Run this cell to view all current cases from the sheet
cases = load_cases_from_sheet()
if not cases:
    print("No cases found in sheet.")
else:
    print(f"{'Case ID':<22} {'Date':<17} {'Queue':<18} {'Status':<12} Sender")
    print("-" * 100)
    for r in cases:
        print(f"{r.get('Case ID',''):<22} {r.get('Date Received',''):<17} {r.get('Queue',''):<18} {r.get('Status',''):<12} {r.get('Sender','')[:40]}")
    print(f"\n{len(cases)} case(s).")

---
## 8. Actions — Assign / Complete / Reclassify

In [ ]:
def assign_case(case_id: str, user: str | None = None):
    user = user or os.environ.get("USER", "unknown")
    now  = datetime.now(TZ).strftime("%d/%m/%Y %H:%M")
    cases = load_cases_from_sheet()
    if not any(r.get("Case ID") == case_id for r in cases):
        print(f"Case not found: {case_id}")
        return
    update_case(case_id, status="In Progress", assigned_to=user, assigned_at=now)
    print(f"Assigned {case_id} to {user}.")

def complete_case(case_id: str):
    now  = datetime.now(TZ).strftime("%d/%m/%Y %H:%M")
    cases = load_cases_from_sheet()
    if not any(r.get("Case ID") == case_id for r in cases):
        print(f"Case not found: {case_id}")
        return
    update_case(case_id, status="Completed", completed_at=now)
    print(f"Completed {case_id}.")

def reclassify_case(case_id: str, queue: str):
    cases = load_cases_from_sheet()
    if not any(r.get("Case ID") == case_id for r in cases):
        print(f"Case not found: {case_id}")
        return
    update_case(case_id, queue=queue)
    print(f"Reclassified {case_id} → {queue}.")

print("Action helpers defined")

---
## 9. Run: Poll Once
Checks Gmail right now, writes new cases to DB, then syncs to Google Sheet.

In [37]:
check_new_emails()

2026-05-24 09:49:03  INFO    === Polling Gmail ===
2026-05-24 09:49:03  INFO    file_cache is only supported with oauth2client<4.0.0
2026-05-24 09:49:04  INFO    Loaded 0 existing message IDs from DB.
2026-05-24 09:49:04  INFO    Found 0 unprocessed messages.
2026-05-24 09:49:04  INFO    file_cache is only supported with oauth2client<4.0.0
2026-05-24 09:49:06  INFO    Synced 0 active cases → 'Cases' tab.
2026-05-24 09:49:07  INFO    Synced 0 archived cases → 'Archive' tab.
2026-05-24 09:49:07  INFO    === Done in 4.3s | New: 0 | Skipped: 0 ===


---
## 10. Run: List Cases
Change `status_filter` to `"New"`, `"In Progress"`, `"Completed"`, or `None` for all.

In [41]:
# remove current existing label

def remove_dispute_labels_from_inbox():
    service = get_gmail_service()

    labels = service.users().labels().list(userId="me").execute().get("labels", [])
    label_map = {label["name"]: label["id"] for label in labels}

    target_names = ["dispute-tracker", "dispute-tracker-processed"]
    remove_label_ids = [label_map[name] for name in target_names if name in label_map]

    if not remove_label_ids:
        print("No matching labels found:", target_names)
        return

    query = "in:inbox (label:dispute-tracker OR label:dispute-tracker-processed)"
    total = 0
    next_page_token = None

    while True:
        response = service.users().messages().list(
            userId="me",
            q=query,
            pageToken=next_page_token,
            maxResults=500,
        ).execute()

        messages = response.get("messages", [])
        if not messages:
            break

        for msg in messages:
            service.users().messages().modify(
                userId="me",
                id=msg["id"],
                body={"removeLabelIds": remove_label_ids},
            ).execute()
            total += 1

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    print(f"Removed dispute labels from {total} inbox messages.")

    remove_dispute_labels_from_inbox()

In [43]:
def sync_from_date(date_str="2026/05/16"):
    service = get_gmail_service()
    query = f"in:anywhere to:{CONFIG['GROUP_EMAIL']} after:{date_str}"
    result = service.users().messages().list(
        userId="me",
        q=query,
        maxResults=500,
    ).execute()
    messages = result.get("messages", [])
    print("Found", len(messages), "messages since", date_str)
    return messages

In [44]:
msgs = sync_from_date("2026/05/16")

2026-05-24 09:58:22  INFO    file_cache is only supported with oauth2client<4.0.0


Found 64 messages since 2026/05/16


In [ ]:
status_filter = None  # "New" | "In Progress" | "Completed" | None

cases = load_cases_from_sheet()
if status_filter:
    rows = [r for r in cases if r.get("Status") == status_filter]
else:
    rows = cases

if not rows:
    print("No cases found.")
else:
    print(f"{'Case ID':<22} {'Date':<17} {'Queue':<18} {'Status':<12} Sender")
    print("-" * 100)
    for r in rows:
        print(f"{r.get('Case ID',''):<22} {r.get('Date Received',''):<17} {r.get('Queue',''):<18} {r.get('Status',''):<12} {r.get('Sender','')[:40]}")
    print(f"\n{len(rows)} case(s).")

### Assign / Complete / Reclassify a case
Fill in the `case_id` and run the relevant cell.

In [39]:
assign_case("CASE-20261003-12345")
# assign_case("CASE-XXXXXXXX-XXXXX", "colleague@grabtaxi.com")

Case not found: CASE-20261003-12345


In [ ]:
complete_case("CASE-XXXXXXXX-XXXXX")

In [ ]:
# Queues: Dispute | Invoice | Update Details | Internal Invoice | Others
reclassify_case("CASE-XXXXXXXX-XXXXX", "Invoice")

---
## 11. Diagnostics

In [ ]:
# ── Health check ─────────────────────────────────────────────────────────────
print("=== Dispute Tracker Diagnostic ===")

print("✅ SLACK_WEBHOOK_URL is set" if SLACK_WEBHOOK_URL else "⚠️  SLACK_WEBHOOK_URL not set")

try:
    cases = load_cases_from_sheet()
    print(f"✅ Sheet OK — {len(cases)} active case(s) in '{CONFIG['SHEET_TAB_NAME']}' tab")
except Exception as e:
    print(f"❌ Sheet error: {e}")

try:
    service = get_gmail_service()
    profile = service.users().getProfile(userId="me").execute()
    print(f"✅ Gmail authenticated as {profile['emailAddress']}")
    query  = f"to:{CONFIG['GROUP_EMAIL']} -label:{CONFIG['PROCESSED_LABEL']}"
    result = service.users().messages().list(userId="me", q=query, maxResults=5).execute()
    print(f"📧 Unprocessed emails (estimate): {result.get('resultSizeEstimate', 0)}")
except Exception as e:
    print(f"❌ Gmail error: {e}")

print(f"\nSheet ID: {CONFIG['SHEET_ID']}")
print("=== Done ===")

In [ ]:
# ── Debug archive (dry run) ───────────────────────────────────────────────────
print("=== Archive Debug ===")
cutoff = datetime.now(TZ) - timedelta(days=CONFIG["ARCHIVE_AFTER_DAYS"])
print(f"Cutoff: cases completed before {cutoff.strftime('%d/%m/%Y')} will be archived")
print(f"Today:  {datetime.now(TZ).strftime('%d/%m/%Y')}\n")

cases = load_cases_from_sheet()
completed = [r for r in cases if r.get("Status") == "Completed"]

if not completed:
    print("No completed cases.")
else:
    would_archive = 0
    for r in completed:
        raw = r.get("Completed At", "")
        completed_dt = _parse_completed_at(raw)
        if completed_dt is None:
            print(f"⚠️  {r.get('Case ID')} — could not parse: {raw!r}")
            continue
        age_days = (datetime.now(TZ) - completed_dt).days
        if completed_dt < cutoff:
            would_archive += 1
            print(f"✅ {r.get('Case ID')} — {raw} ({age_days}d ago) → WOULD ARCHIVE")
        else:
            print(f"⏳ {r.get('Case ID')} — {raw} ({age_days}d ago) → keep")
    print(f"\nTotal completed: {len(completed)} | Would archive: {would_archive}")

print("=== Done ===")

In [ ]:
# ── Sheet row health check ────────────────────────────────────────────────────
print("=== Sheet Row Health Check ===")

cases = load_cases_from_sheet()
empty_rows = [r for r in cases if not r.get("Case ID", "").strip()]
in_progress = [r for r in cases if r.get("Status") == "In Progress"]

print(f"Total rows in sheet:      {len(cases)}")
print(f"Empty Case ID rows:       {len(empty_rows)}")
print(f"Total In Progress:        {len(in_progress)}")
for r in empty_rows:
    print(f"⚠️  Empty case_id row: Email Link={r.get('Email Link','')!r}")
print("=== Done ===")

---
## 12. Tests

In [ ]:
# ── Test classification ───────────────────────────────────────────────────────
TEST_CASES = [
    ("Khiếu nại chênh lệch hóa đơn tháng 4", "chênh lệch trong bảng kê",     "Dispute"),
    ("Yêu cầu cập nhật thông tin công ty",    "thông tin không chính xác",     "Update Details"),
    ("Request Invoice for March 2025",         "chưa nhận được hóa đơn",       "Invoice"),
    ("Internal Invoice Q1 Settlement",         "internal invoice between ents", "Internal Invoice"),
    ("General inquiry about payment",          "question about my payment",     "Others"),
]

passed = 0
for subject, body, expected in TEST_CASES:
    got = classify_email(subject, body)
    ok  = got == expected
    passed += ok
    note = "" if ok else f" (expected {expected})"
    print(f"[{'PASS' if ok else 'FAIL'}] \"{subject[:45]}\" → {got}{note}")

print(f"\n{passed}/{len(TEST_CASES)} passed.")

In [ ]:
pip install -r requirements.txt
